In [1]:
import weaviate
from weaviate.classes.config  import Configure,Property, DataType
import requests, json
import base64
from pathlib import Path
import os


In [2]:

client = weaviate.connect_to_local(
    host="172.17.0.2",  
    port=8080,
    grpc_port=50051,
)

In [3]:
collections = client.collections.delete("Grounded_nomic_full")
print(collections)

None


In [4]:
schema = {
    "class": "Grounded_nomic_full",
    "vectorConfig": {
        "default": {
            "vectorIndexType": "hnsw",
            "vectorizer": {
                "text2vec-ollama": {
                    "apiEndpoint": "http://172.17.0.4:11434",
                    "model": "nomic-embed-text",
                    "vectorizeClassName": True
                }
            }
        }
    },
    "moduleConfig": {
        "generative-ollama": {
            "apiEndpoint": "http://172.17.0.4:11434",
            "model": "llama3.2"
        }
    },
    "properties": [
        {"name": "type",        "dataType": ["text"],  "moduleConfig": {"text2vec-ollama": {"skip": True,  "vectorizePropertyName": False}}},
        {"name": "page",        "dataType": ["int"],   "moduleConfig": {"text2vec-ollama": {"skip": True,  "vectorizePropertyName": False}}},
        {"name": "description", "dataType": ["text"],  "moduleConfig": {"text2vec-ollama": {"skip": False, "vectorizePropertyName": False}}},
        {"name": "text",        "dataType": ["text"],  "moduleConfig": {"text2vec-ollama": {"skip": False, "vectorizePropertyName": False}}},
        {"name": "trace",       "dataType": ["text"],  "moduleConfig": {"text2vec-ollama": {"skip": False, "vectorizePropertyName": False}}},
        {"name": "filename",    "dataType": ["text"],  "moduleConfig": {"text2vec-ollama": {"skip": False, "vectorizePropertyName": False}}},
        {"name": "image",       "dataType": ["blob"],  "moduleConfig": {"text2vec-ollama": {"skip": True,  "vectorizePropertyName": False}}}
    ]
}

r = requests.post("http://172.17.0.2:8080/v1/schema", json=schema)

In [5]:
questions = client.collections.use("Grounded_nomic_full")
r = requests.get("http://172.17.0.2:8080/v1/schema/Grounded_nomic_full")
for p in r.json()["properties"]:
    print(p["name"], p["moduleConfig"])

type {'text2vec-ollama': {'skip': True, 'vectorizePropertyName': False}}
page {'text2vec-ollama': {'skip': True, 'vectorizePropertyName': False}}
description {'text2vec-ollama': {'skip': False, 'vectorizePropertyName': False}}
text {'text2vec-ollama': {'skip': False, 'vectorizePropertyName': False}}
trace {'text2vec-ollama': {'skip': False, 'vectorizePropertyName': False}}
filename {'text2vec-ollama': {'skip': False, 'vectorizePropertyName': False}}
image {'text2vec-ollama': {'skip': True, 'vectorizePropertyName': False}}


In [6]:
for filename in sorted(os.listdir("../clean_chunks")):
    if filename.endswith(".json"):
        filepath = os.path.join("../clean_chunks", filename)
            
    with open(filepath, 'r') as f:
        data = json.load(f)
    print(f"Importing data from {filename} with {len(data)} entries...")
    with questions.batch.fixed_size(batch_size=200) as batch:
        for d in data:
            properties = {
                    "type": d["block_type"],
                    "page": d["page"],
                    "description": d["description"],
                    "text": d["text"],
                    "trace": d["trace"],
                    "filename": d["filename"],
                #    "Image": list(d["images"].values())[0] if d["images"] else None,
                }

            # Handle image properly
            if d["images"]:
                properties["image"] = d["images"] 
        
            batch.add_object(properties)
            
            if batch.number_errors > 10:
                print("Batch import stopped due to excessive errors.")
                break

    failed_objects = questions.batch.failed_objects
    if failed_objects:
        print(f"Number of failed imports: {len(failed_objects)}")
        print(f"failed on flename:{filepath}")
        print(f"First failed object: {failed_objects[0]}")
        
    print(f"Finished importing data from {filename}")

client.close()  # Free up resources

Importing data from O-RAN-WG1-CCIN-TR-R004-v01.00_cleaned.json with 609 entries...
Finished importing data from O-RAN-WG1-CCIN-TR-R004-v01.00_cleaned.json
Importing data from O-RAN-WG6.AppLCM-Deployment-R003-v02.00_cleaned.json with 376 entries...
Finished importing data from O-RAN-WG6.AppLCM-Deployment-R003-v02.00_cleaned.json
Importing data from O-RAN.SFG.Non-RT-RIC-Security-TR-v01.00_cleaned.json with 346 entries...
Finished importing data from O-RAN.SFG.Non-RT-RIC-Security-TR-v01.00_cleaned.json
Importing data from O-RAN.SuFG.CE-v01.00_cleaned.json with 163 entries...
Finished importing data from O-RAN.SuFG.CE-v01.00_cleaned.json
Importing data from O-RAN.SuFG.TR.NES-Analysis-R004-v01.01_cleaned.json with 357 entries...
Finished importing data from O-RAN.SuFG.TR.NES-Analysis-R004-v01.01_cleaned.json
Importing data from O-RAN.TIFG.CGofOTIC.0-v06.00_cleaned.json with 241 entries...
Finished importing data from O-RAN.TIFG.CGofOTIC.0-v06.00_cleaned.json
Importing data from O-RAN.TIFG.E

{'message': 'Failed to send 10 in a batch of 200', 'errors': {'connection to Ollama API failed with error: the input length exceeds the context length'}}
{'message': 'Failed to send 10 objects in a batch of 200. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}


Number of failed imports: 10
failed on flename:../clean_chunks/O-RAN.WG11.TS.SRCS.0-R004-v13.00_cleaned.json
First failed object: ErrorObject(message='connection to Ollama API failed with error: the input length exceeds the context length', object_=BatchObject(collection='Grounded_nomic_full', properties={'type': 'TableOfContents', 'page': 2, 'description': '', 'text': '| Modal verbs terminologyForew/ord 4  \n2 References 5 2.1 Normative references 5 2.2 Informative references 8 3 Definition of terms, symbols and abbreviations 10 3.1 Terms 10 3.2 Symbols 14 3.3 Abbreviations 15 4 Objectives and scope 16 4.1 Objectives 16 4.2 Perimeter 17 4.2.1 Void 18 4.2.2 Void 18 4.2.3 Void 18 4.2.4.2.1 Void 18 5.0 Introduction 18 5.1 O-RAN Architecture Elements 18 5.1 O-RAN Architecture Elements 18 5.1.1 Service Management and Orchestration \\(SMO\\) 18 5.1.2 Non-RT RIC and Apps 24 5.1.3 Near-RT RIC and Apps 25 5.1.4 O-CU-CP/UP 31 5.1.5 O-DUModal verbs terminology 4  \n2.1 Normative references 5 2.2

{'message': 'Failed to send 10 in a batch of 200', 'errors': {'connection to Ollama API failed with error: the input length exceeds the context length'}}
{'message': 'Failed to send 10 objects in a batch of 200. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}


Number of failed imports: 10
failed on flename:../clean_chunks/O-RAN.WG11.TS.STS-R004-v11.00_cleaned.json
First failed object: ErrorObject(message='connection to Ollama API failed with error: the input length exceeds the context length', object_=BatchObject(collection='Grounded_nomic_full', properties={'type': 'Picture', 'page': 4, 'description': '', 'text': '', 'trace': 'Contents', 'filename': 'O-RAN.WG11.TS.STS-R004-v11.00', 'image': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL/2wBDAQkJCQwLDBgNDRgyIRwhMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjL/wAARCABNAOsDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQ

...

Number of failed imports: 10
failed on flename:../clean_chunks/O-RAN.WG3.TS.E2AP-R004-v08.00_cleaned.json
First failed object: ErrorObject(message='connection to Ollama API failed with error: the input length exceeds the context length', object_=BatchObject(collection='Grounded_nomic_full', properties={'type': 'Picture', 'page': 4, 'description': '', 'text': '', 'trace': 'Contents', 'filename': 'O-RAN.WG3.TS.E2AP-R004-v08.00', 'image': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL/2wBDAQkJCQwLDBgNDRgyIRwhMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjL/wAARCABNAOoDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQ

{'message': 'Failed to send 10 in a batch of 200', 'errors': {'connection to Ollama API failed with error: the input length exceeds the context length'}}
{'message': 'Failed to send 10 objects in a batch of 200. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}


Number of failed imports: 10
failed on flename:../clean_chunks/O-RAN.WG7.OMAC-HRD.0-R004-v04.00_cleaned.json
First failed object: ErrorObject(message='connection to Ollama API failed with error: the input length exceeds the context length', object_=BatchObject(collection='Grounded_nomic_full', properties={'type': 'Text', 'page': 55, 'description': '', 'text': 'Figure 2.3.2-14 Example 64T64R JESD204 Configuration', 'trace': '1 a. Accelerator Requirements --> 9 Table 2.3.2-11 LNA RF Specifications --> O-RAN.WG7.OMAC-HRD.0-R004-v04.00', 'filename': 'O-RAN.WG7.OMAC-HRD.0-R004-v04.00'}, references=None, uuid='34111712-315a-4c35-8214-b3675f048982', vector=None, tenant=None, index=710, retry_count=0), original_uuid='34111712-315a-4c35-8214-b3675f048982')
Finished importing data from O-RAN.WG7.OMAC-HRD.0-R004-v04.00_cleaned.json
Importing data from O-RAN.WG7.OMC-HAR.0-v01.00_cleaned.json with 471 entries...
Finished importing data from O-RAN.WG7.OMC-HAR.0-v01.00_cleaned.json
Importing data fro

{'message': 'Failed to send 10 in a batch of 200', 'errors': {'connection to Ollama API failed with error: the input length exceeds the context length'}}
{'message': 'Failed to send 10 objects in a batch of 200. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}


Number of failed imports: 10
failed on flename:../clean_chunks/O-RAN.WG8.TS.IOT.0-R005-v14.00_cleaned.json
First failed object: ErrorObject(message='connection to Ollama API failed with error: the input length exceeds the context length', object_=BatchObject(collection='Grounded_nomic_full', properties={'type': 'TableOfContents', 'page': 8, 'description': '', 'text': '7.51.1 | Reference Requirement | 7.51.2 | Test Setup and Configuration | 7.52.1 | Test Purpose | 7.52.1 | Test Purpose | 7.52.1 | Test Purpose | 7.52.2 | Reference Requirement | 7.52.3 | Initial Conditions | 7.52.4 | Test Setup and Configuration | 7.52.5 | Test Setup and Configuration | 7.52.1 | Test Purpose | 7.52.2 | Initial Conditions | 7.52.3 | Test Setup and Configuration | 7.53.3 | ORAN WG8.IOT.052: Verify HO preparation failure at target O-DU during inter-O-DU h within an O-CU | 7.53.1 | Test Purpose | 7.53.2 | Reference Requirement | 7.53.3 | Test Purpose | 7.53.3 | Test Purpose | 7.54.1 | Test Setup and Configura

In [18]:
import weaviate
import json
from weaviate.classes.query import Filter
client = weaviate.connect_to_local(
    host="172.17.0.2",  
    port=8080,
    grpc_port=50051,
)
questions = client.collections.use("Grounded_nomic_full")

#my_filter = Filter.by_property("image").is_none(False)


response = questions.query.near_text(

    query=""" 

how is a producded iniated by E2 Nodes        """,
    limit=10
    
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=2))

client.close()  # Free up resources

{
  "description": "",
  "page": 7,
  "text": "Managed Element: The definition of a Managed Element \\(ME\\) is given in 3GPP TS 28.622 \\[2\\]\\[2\\], clause 4.3.3.",
  "trace": "3.1 Terms",
  "filename": "O-RAN.WG1.TS.OAD-R004-v15.00",
  "type": "Text"
}
{
  "description": "",
  "page": 7,
  "filename": "O-RAN.WG1.TS.OAD-R004-v15.00",
  "text": "Managed Function: The definition of a Managed Function \\(MF\\) is given in 3GPP TS 28.622 \\[2\\], clause 4.3.4.",
  "trace": "3.1 Terms",
  "type": "Text"
}
{
  "description": "",
  "page": 45,
  "text": "The definition of a Network Element \\(NE\\) is given in 3GPP TS 21.905 \\[i.1\\], clause 3.N. In the O-RAN OAM Architecture, an O-RAN Network Element is a logical entity and aggregates O-RAN Network Functions. A set of Management Functions in the NE produce Management Services toward the SMO.",
  "trace": "5.1 Architectural Principles --> 5.3.1 Architectural Building Blocks --> 5.3.1.2 O-RAN Network Elements",
  "filename": "O-RAN.WG10.TS

In [19]:
client.close()